# Load libraries

In [ ]:
# LOAD LIBRARIES
import pandas as pd, numpy as np
from tqdm import tqdm
import gc
import cudf, cupy
from cuml.cluster import KMeans
import matplotlib.pyplot as plt
print('RAPIDS version',cudf.__version__)

# Load data

In [ ]:
# LOAD TRAIN
train = cudf.read_csv('../input/ventilator-pressure-prediction/train.csv')
test = cudf.read_csv('../input/ventilator-pressure-prediction/test.csv')

test['breath_id'] = test['breath_id'] + 200000

print(train.shape)
print(test.shape)

all_df = cudf.concat([train, test])
print(all_df.shape)

# Data preparation

In [ ]:
exhale = 80-all_df.groupby('breath_id')[['u_out']].agg('sum')
length = all_df.groupby('breath_id')[['time_step']].agg('max')
print('All_df shape:',all_df.shape)
all_df.head()

In [ ]:
# CONVERT TRAIN TO SERIES
series = all_df.groupby('breath_id').collect().reset_index()
for k in range(80): series[f'x_{k}'] = series.u_in.list.get(k)
# for k in range(80): series[f'y_{k}'] = series.pressure.list.get(k)
for k in range(80): series[f'z_{k}'] = 1-series.u_out.list.get(k)
series.R = series.R.list.get(0)
series.C = series.C.list.get(0)
series = series.drop(['id','time_step','u_in','u_out','pressure'],axis=1)
series = series.merge(exhale,on='breath_id',how='left')
series = series.merge(length,on='breath_id',how='left')
series = series.rename({'time_step':'time_length','u_out':'expire'},axis=1)
series = series.sort_values('breath_id').reset_index(drop=True)

print('Train as series shape:', series.shape )
print('Min inhale length=', series['expire'].min(),',Max inhale length=', series['expire'].max(),
      'Max breath length=',series['time_length'].max() )
series.head()

# KMeans features

In [ ]:
# FIRST 3 COLUMNS OF TRAIN SERIES DATAFRAME NEED TO BE IGNORED
IGNORE = 3
# MORE NEIGHBORS THAN WE NEED FOR ALL DISPLAYS
NEIGHBORS = 500
# HOW MANY TIME_STEPS TO USE FOR KNN COMPARISON
TIME_STEPS = 80
# HOW MANY INITIAL TIME_STEPS TO SKIP FOR KNN COMPARISON
SKIP = 0 

In [ ]:
model = KMeans(n_clusters=1000)
model.fit(series.iloc[:,IGNORE+SKIP:IGNORE+TIME_STEPS+SKIP])

In [ ]:
series[f'kmeans{1000}_cluster'] = model.labels_
idx = series[f'kmeans{1000}_cluster'].value_counts().index.values

temp = series[['breath_id', f'kmeans{1000}_cluster']].copy()

all_df = all_df.merge(temp, on='breath_id', how="left")
all_df = all_df.sort_values(by=['breath_id','id'])
all_df = all_df.reset_index(drop=True)

In [ ]:
all_df.head()

In [ ]:
temp = all_df.groupby(['kmeans1000_cluster'])['breath_id'].collect().reset_index()
temp = temp.to_pandas()
temp['breath_id'] = temp['breath_id'].apply(lambda x: list(set(x)))


kmeansAllDict = dict(zip(temp['kmeans1000_cluster'].tolist(),temp['breath_id'].tolist()))

temp.head()

In [ ]:
# CONVERT TRAIN TO series_trn
series_trn = train.groupby('breath_id').collect().reset_index()
for k in range(80): series_trn[f'x_{k}'] = series_trn.u_in.list.get(k)
for k in range(80): series_trn[f'y_{k}'] = series_trn.pressure.list.get(k)
for k in range(80): series_trn[f'z_{k}'] = 1-series_trn.u_out.list.get(k)
series_trn.R = series_trn.R.list.get(0)
series_trn.C = series_trn.C.list.get(0)
series_trn = series_trn.drop(['id','time_step','u_in','u_out','pressure'],axis=1)
series_trn = series_trn.merge(exhale,on='breath_id',how='left')
series_trn = series_trn.merge(length,on='breath_id',how='left')
series_trn = series_trn.rename({'time_step':'time_length','u_out':'expire'},axis=1)
series_trn = series_trn.sort_values('breath_id').reset_index(drop=True)

print('Train as series_trn shape:', series_trn.shape )
print('Min inhale length=', series_trn['expire'].min(),',Max inhale length=', series_trn['expire'].max(),
      'Max breath length=',series_trn['time_length'].max() )
series_trn.head()

In [ ]:
model = KMeans(n_clusters=1000)
model.fit(series.iloc[:,IGNORE+SKIP:IGNORE+TIME_STEPS+SKIP])

series_trn[f'kmeans{1000}_cluster_trn'] = model.labels_
idx = series_trn[f'kmeans{1000}_cluster_trn'].value_counts().index.values

temp = series_trn[['breath_id', f'kmeans{1000}_cluster_trn']].copy()

train = train.merge(temp, on='breath_id', how="left")
train = train.sort_values(by=['breath_id','id'])
train = train.reset_index(drop=True)

In [ ]:
train['kmeans1000_cluster_trn'].value_counts(ascending=False).index[0]

In [ ]:
all_df[all_df['breath_id']==200000]['kmeans1000_cluster'].unique()[0]

In [ ]:
lst =[1,34,34,2]
max(set(lst), key=lst.count)

In [ ]:
# Assign train cluster to test
test[f'kmeans{1000}_cluster_trn'] = -1

testLst = test.to_pandas()['breath_id'].unique().tolist()

testTrnClusterDict = dict()

for breath in tqdm(testLst):
    
    cluster = all_df[all_df['breath_id']==breath]['kmeans1000_cluster'].unique()[0]
    
    trnclusts = []
    trnLst = [brth for brth in kmeansAllDict[cluster] if brth not in testLst]
    
    testTrnClusterDict[breath] = train[train['breath_id'].isin(trnLst)]['kmeans1000_cluster_trn'].value_counts(ascending=False).index[0]

In [ ]:

for num in tqdm([x*10 for x in [1,10,100,1000,10000]]):
    model = KMeans(n_clusters=num)
    model.fit(series.iloc[:,IGNORE+SKIP:IGNORE+TIME_STEPS+SKIP])
    series[f'kmeans{num}_cluster'] = model.labels_
    idx = series[f'kmeans{num}_cluster'].value_counts().index.values
      
    temp = series[['breath_id', f'kmeans{num}_cluster']].copy()

    all_df = all_df.merge(temp, on='breath_id', how="left")
    all_df = all_df.sort_values(by=['breath_id','id'])
    all_df = all_df.reset_index(drop=True)
    
    all_df['breath_steps'] = all_df[['breath_id','R']].groupby('breath_id')['R'].cumcount().reset_index(drop=True)+1
    all_df['R__C'] = all_df["R"].astype(str) + '__' + all_df["C"].astype(str)

    temp1 = all_df[[f'kmeans{num}_cluster','R__C','u_in','breath_steps']].groupby([f'kmeans{num}_cluster','R__C','breath_steps']).mean()["u_in"].reset_index().rename(columns={'u_in':f'u_in_grp_kmeans{num}_R__C_steps_mean'}).copy()
    temp2 = all_df[[f'kmeans{num}_cluster','R__C','u_in','breath_steps']].groupby([f'kmeans{num}_cluster','R__C','breath_steps']).max()["u_in"].reset_index().rename(columns={'u_in':f'u_in_grp_kmeans{num}_R__C_steps_max'}).copy()
    temp3 = all_df[[f'kmeans{num}_cluster','R__C','u_in','breath_steps']].groupby([f'kmeans{num}_cluster','R__C','breath_steps']).min()["u_in"].reset_index().rename(columns={'u_in':f'u_in_grp_kmeans{num}_R__C_steps_min'}).copy()


    temp = temp1.merge(temp2, on=[f'kmeans{num}_cluster',"R__C","breath_steps"], how="left")
    temp = temp.merge(temp3, on=[f'kmeans{num}_cluster',"R__C","breath_steps"], how="left")

    del temp1, temp2, temp3
    _ = gc.collect()

    all_df = all_df.merge(temp, on=[f'kmeans{num}_cluster',"R__C","breath_steps"], how="left")

    del temp
    _ = gc.collect()

    all_df[f'u_in_grp_kmeans{num}_R__C_steps_mean_diff'] = all_df["u_in"] - all_df[f"u_in_grp_kmeans{num}_R__C_steps_mean"]
    all_df[f'u_in_grp_kmeans{num}_R__C_steps_max_diff'] = all_df["u_in"] - all_df[f"u_in_grp_kmeans{num}_R__C_steps_max"]
    all_df[f'u_in_grp_kmeans{num}_R__C_steps_min_diff'] = all_df["u_in"] - all_df[f"u_in_grp_kmeans{num}_R__C_steps_min"]
    
    
    

In [ ]:
all_df = all_df.to_pandas()
all_df["time_passed"] = all_df.groupby("breath_id")["time_step"].diff()
all_df = cudf.from_pandas(all_df)

for num in tqdm([x*10 for x in [1,10,100,1000,10000]]):
    

    temp1 = all_df[[f'kmeans{num}_cluster','R__C','time_passed','breath_steps']].groupby([f'kmeans{num}_cluster','R__C','breath_steps']).mean()["time_passed"].reset_index().rename(columns={'time_passed':f'time_passed_grp_kmeans{num}_R__C_steps_mean'}).copy()
    temp2 = all_df[[f'kmeans{num}_cluster','R__C','time_passed','breath_steps']].groupby([f'kmeans{num}_cluster','R__C','breath_steps']).max()["time_passed"].reset_index().rename(columns={'time_passed':f'time_passed_grp_kmeans{num}_R__C_steps_max'}).copy()
    temp3 = all_df[[f'kmeans{num}_cluster','R__C','time_passed','breath_steps']].groupby([f'kmeans{num}_cluster','R__C','breath_steps']).min()["time_passed"].reset_index().rename(columns={'time_passed':f'time_passed_grp_kmeans{num}_R__C_steps_min'}).copy()


    temp = temp1.merge(temp2, on=[f'kmeans{num}_cluster',"R__C","breath_steps"], how="left")
    temp = temp.merge(temp3, on=[f'kmeans{num}_cluster',"R__C","breath_steps"], how="left")

    del temp1, temp2, temp3
    _ = gc.collect()

    all_df = all_df.merge(temp, on=[f'kmeans{num}_cluster',"R__C","breath_steps"], how="left")

    del temp
    _ = gc.collect()

    all_df[f'time_passed_grp_kmeans{num}_R__C_steps_mean_diff'] = all_df["time_passed"] - all_df[f"time_passed_grp_kmeans{num}_R__C_steps_mean"]
    all_df[f'time_passed_grp_kmeans{num}_R__C_steps_max_diff'] = all_df["time_passed"] - all_df[f"time_passed_grp_kmeans{num}_R__C_steps_max"]
    all_df[f'time_passed_grp_kmeans{num}_R__C_steps_min_diff'] = all_df["time_passed"] - all_df[f"time_passed_grp_kmeans{num}_R__C_steps_min"]

In [ ]:
all_df = all_df.sort_values(by=['breath_id','id'])
all_df = all_df.reset_index(drop=True)

In [ ]:
all_df.head()

In [ ]:
del train, test, series
_ = gc.collect()

In [ ]:
train = all_df[all_df['breath_id']<200000].copy()
test = all_df[all_df['breath_id']>=200000].copy()

print(train.shape)
print(test.shape)

In [ ]:
train.to_feather("GBVPP_KNN_train.feather")
test.to_feather("GBVPP_KNN_test.feather")

In [ ]:
# breath_dict = dict(zip(series.index.to_pandas().tolist(),series['breath_id'].to_pandas().tolist()))

# from operator import itemgetter

# itemgetter(1, 3, 2, 5)(breath_dict)